In [198]:
# Manipulacion de datos
import pandas as pd
import numpy as np
import datetime
pd.set_option('display.max_columns', 200)
import json
import os
import unicodedata

import geopandas as gpd
from shapely.geometry import Point

from pathlib import Path

In [199]:
# Hacer joins

In [200]:
df_agricolas = pd.read_csv('/Users/jaydymarchan/Desktop/causalidad/data/02_processed/datos_agricolas_limpios.csv', encoding='utf-8')
df_lluvias = pd.read_csv('/Users/jaydymarchan/Desktop/causalidad/data/02_processed/datos_lluvia.csv', encoding='utf-8')
df_sequia = pd.read_csv('/Users/jaydymarchan/Desktop/causalidad/data/02_processed/datos_sequia.csv', encoding='utf-8')
df_tmp_min = pd.read_csv('/Users/jaydymarchan/Desktop/causalidad/data/02_processed/datos_temp_min.csv', encoding='utf-8')

In [201]:
# Eliminar columnas no requeridas
df_agricolas = df_agricolas.drop(columns=['idcultivo', 'idmodalidad', 'idddr', 'nomunidad', 'nomcultivo'])

# Reordenar (y quedarnos solo con) las columnas en el orden solicitado
orden_columnas = [
    'anio', 'idestado', 'idmunicipio', 'nomcicloproductivo', 'nomcader',
    'nomestado', 'nommunicipio', 'nommodalidad', 'cosechada', 'precio', 'rendimiento',
    'sembrada', 'siniestrada', 'valorproduccion', 'volumenproduccion',
]
df_agricolas = df_agricolas[orden_columnas]

# Codificar el ciclo productivo: Primavera-Verano -> PV, resto -> OI
df_agricolas['nomcicloproductivo'] = np.where(
    df_agricolas['nomcicloproductivo'] == 'Primavera-Verano', 'PV', 'OI',
)

df_agricolas

,anio,idestado,idmunicipio,nomcicloproductivo,nomcader,nomestado,nommunicipio,nommodalidad,cosechada,precio,rendimiento,sembrada,siniestrada,valorproduccion,volumenproduccion
0,2016,1,1,PV,Aguascalientes,Aguascalientes,Aguascalientes,Riego,300.0,3473.32,7.49,300.0,0.0,7805071.04,2247.15
1,2016,1,1,PV,Aguascalientes,Aguascalientes,Aguascalientes,Temporal,5032.0,3359.06,0.67,5045.0,13.0,11255639.02,3350.83
2,2016,1,5,PV,Aguascalientes,Aguascalientes,Jesús María,Riego,266.0,3734.10,7.26,266.0,0.0,7209986.99,1930.85
3,2016,1,5,PV,Aguascalientes,Aguascalientes,Jesús María,Temporal,630.0,3345.10,0.58,630.0,0.0,1222332.99,365.41
4,2016,1,10,PV,Aguascalientes,Aguascalientes,El Llano,Riego,197.0,3509.84,7.48,197.0,0.0,5174206.13,1474.20
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
55848,2024,32,1,PV,Jalpa,Zacatecas,Apozol,Riego,85.0,6152.94,7.75,85.0,0.0,4053249.23,658.75
55849,2024,32,1,PV,Jalpa,Zacatecas,Apozol,Temporal,720.0,6055.56,2.80,720.0,0.0,12208008.96,2016.00
55850,2024,32,19,OI,Jalpa,Zacatecas,Jalpa,Riego,2.0,7000.00,7.50,2.0,0.0,105000.00,15.00
55851,2024,32,19,PV,Jalpa,Zacatecas,Jalpa,Riego,221.0,6165.61,7.75,221.0,0.0,10560148.53,1712.75


In [202]:
df_agricolas[(df_agricolas['nommunicipio']=='Aguascalientes') & (df_agricolas['nomestado']=='Aguascalientes') & (df_agricolas['anio']==2016)    ]

,anio,idestado,idmunicipio,nomcicloproductivo,nomcader,nomestado,nommunicipio,nommodalidad,cosechada,precio,rendimiento,sembrada,siniestrada,valorproduccion,volumenproduccion
0,2016,1,1,PV,Aguascalientes,Aguascalientes,Aguascalientes,Riego,300.0,3473.32,7.49,300.0,0.0,7805071.04,2247.15
1,2016,1,1,PV,Aguascalientes,Aguascalientes,Aguascalientes,Temporal,5032.0,3359.06,0.67,5045.0,13.0,11255639.02,3350.83


In [203]:
# ¿Cómo se calcula el rendimiento?
#   rendimiento (t/ha) = volumenproduccion / superficie cosechada   (definición SIAP)
#   'cosechada' YA es superficie neta: en el cierre agrícola  sembrada = cosechada + siniestrada
#   (se cumple exacto en los datos), asi que NO hay que volver a restar 'siniestrada'.
assert (df_agricolas['sembrada'] - df_agricolas['cosechada'] - df_agricolas['siniestrada']).abs().max() < 1.0
_r = df_agricolas['volumenproduccion'] / df_agricolas['cosechada'].replace(0, np.nan)
print('rendimiento recalculado == columna rendimiento de SIAP:',
      bool(np.allclose(_r.dropna(), df_agricolas.loc[_r.notna(), 'rendimiento'], atol=0.02)))
print(df_agricolas['rendimiento'].describe().round(2).to_string())


rendimiento recalculado == columna rendimiento de SIAP: True
count    55834.00
mean         2.94
std          2.36
min          0.00
25%          1.21
50%          2.37
75%          3.80
max         15.30


In [204]:
# Agrupacion a nivel anio-municipio-ciclo-modalidad (Riego / Temporal)
llaves = ['anio', 'idestado', 'idmunicipio', 'nomcicloproductivo', 'nommodalidad', 'nomestado', 'nommunicipio']

df_agricolas_agg = (
    df_agricolas
    .groupby(llaves, as_index=False)
    .agg(
        volumenproduccion=('volumenproduccion', 'sum'),
        cosechada=('cosechada', 'sum'),
        siniestrada=('siniestrada', 'sum'),
        sembrada=('sembrada', 'sum'),
        valorproduccion=('valorproduccion', 'sum'),
        precio_mean=('precio', 'mean'),
        precio_total=('precio', 'sum'),

    )
)

# Rendimiento (t/ha) = volumen producido / superficie cosechada  (definición SIAP).
# 'cosechada' ya excluye la superficie siniestrada; donde cosechada == 0 -> 0.
df_agricolas_agg['rendimiento_real'] = np.where(
    df_agricolas_agg['cosechada'] == 0,
    0.0,
    df_agricolas_agg['volumenproduccion'] / df_agricolas_agg['cosechada'],
)

# Tasa de siniestro (superficie siniestrada / sembrada) -> mide qué tan chica/confiable queda
# la superficie efectivamente cosechada. NO se usa como confusor (es posterior a la sequía);
# sirve solo para decidir qué filas quedan fuera del resultado Y (ver 03_EDA / 04_inferencia_causal).
df_agricolas_agg['tasa_siniestro'] = np.where(
    df_agricolas_agg['sembrada'] == 0, 0.0,
    df_agricolas_agg['siniestrada'] / df_agricolas_agg['sembrada'],
)

df_agricolas_agg

,anio,idestado,idmunicipio,nomcicloproductivo,nommodalidad,nomestado,nommunicipio,volumenproduccion,cosechada,siniestrada,sembrada,valorproduccion,precio_mean,precio_total,rendimiento_real,tasa_siniestro
0,2013,1,1,PV,Riego,Aguascalientes,Aguascalientes,1699.00,230.0,0.0,230.0,6075657.98,3576.02,3576.02,7.386957,0.000000
1,2013,1,1,PV,Temporal,Aguascalientes,Aguascalientes,3944.60,5923.0,0.0,5923.0,14865659.01,3768.61,3768.61,0.665980,0.000000
2,2013,1,2,PV,Riego,Aguascalientes,Asientos,3734.00,582.0,0.0,582.0,14358611.58,3845.37,3845.37,6.415808,0.000000
3,2013,1,2,PV,Temporal,Aguascalientes,Asientos,2988.00,4728.0,1160.0,5888.0,10458000.00,3500.00,3500.00,0.631980,0.197011
4,2013,1,3,PV,Riego,Aguascalientes,Calvillo,290.00,57.0,0.0,57.0,1164999.60,4017.24,4017.24,5.087719,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
53679,2024,32,56,PV,Riego,Zacatecas,Zacatecas,6803.40,870.0,0.0,870.0,53645200.00,7550.00,15100.00,7.820000,0.000000
53680,2024,32,56,PV,Temporal,Zacatecas,Zacatecas,574.00,700.0,0.0,700.0,3478440.00,6060.00,6060.00,0.820000,0.000000
53681,2024,32,57,PV,Riego,Zacatecas,Trancoso,4232.80,520.0,0.0,520.0,32486740.00,7450.00,14900.00,8.140000,0.000000
53682,2024,32,57,PV,Temporal,Zacatecas,Trancoso,555.75,975.0,0.0,975.0,3396627.29,6111.79,6111.79,0.570000,0.000000


In [205]:
# Corregir nombre_entidad y NOMGEO de las tablas de clima usando la CVE como llave.
# Fuente de verdad: la tabla agricola, que es internamente consistente
# (cada idestado -> 1 nomestado; cada (idestado, idmunicipio) -> 1 nommunicipio).
import unicodedata
import re

ANIOS = range(2016, 2025)


def _norm(s):
    s = unicodedata.normalize('NFKD', str(s)).encode('ascii', 'ignore').decode('ascii')
    return re.sub(r'\s+', ' ', re.sub(r'[^a-z0-9 ]', ' ', s.lower())).strip()


# --- catalogo oficial CVE -> nombre ---
_moda = lambda s: s.mode().iat[0]
CAT_ESTADO = df_agricolas.groupby('idestado')['nomestado'].agg(_moda)
CAT_MUNICIPIO = df_agricolas.groupby(['idestado', 'idmunicipio'])['nommunicipio'].agg(_moda)

_KEY = ['idestado', 'idmunicipio', 'anio', 'nomcicloproductivo']


def corregir_clima(df):
    d = df.rename(columns={'CVE_ENT': 'idestado', 'CVE_MUN': 'idmunicipio'}).copy()
    d['idestado'] = d['idestado'].astype(int)
    d['idmunicipio'] = d['idmunicipio'].astype(int)

    # Ante duplicados por CVE+anio+ciclo, conservar la fila cuya nombre_entidad
    # ORIGINAL concuerda con la CVE (la medicion legitima; la otra esta mal capturada).
    est_correcto = d['idestado'].map(CAT_ESTADO)
    d['_ok'] = (d['nombre_entidad'].map(_norm) == est_correcto.map(_norm)).astype(int)
    d = (d.sort_values('_ok', ascending=False)
           .drop_duplicates(_KEY, keep='first')
           .drop(columns='_ok'))

    # Sobrescribir los nombres desde el catalogo (guardando el original para auditar)
    d['nombre_entidad_orig'] = d['nombre_entidad']
    d['NOMGEO_orig'] = d['NOMGEO']
    d['nombre_entidad'] = d['idestado'].map(CAT_ESTADO).fillna(d['nombre_entidad'])
    _mun = d.set_index(['idestado', 'idmunicipio']).index.map(CAT_MUNICIPIO)
    d['NOMGEO'] = pd.Series(_mun, index=d.index).fillna(d['NOMGEO'])
    return d


df_lluvias = corregir_clima(df_lluvias)
df_tmp_min = corregir_clima(df_tmp_min)

# Verificacion: cuantos nombres se corrigieron y que no queden duplicados
for _n, _d in [('lluvia', df_lluvias), ('temp_min', df_tmp_min)]:
    _ent = (_d['nombre_entidad'].map(_norm) != _d['nombre_entidad_orig'].map(_norm)).sum()
    _geo = (_d['NOMGEO'].map(_norm) != _d['NOMGEO_orig'].map(_norm)).sum()
    _dup = _d.duplicated(_KEY).sum()
    print(f'{_n}: entidad corregida en {_ent} filas | NOMGEO en {_geo} filas | duplicados restantes {_dup}')

df_lluvias[df_lluvias['nombre_entidad'].map(_norm) != df_lluvias['nombre_entidad_orig'].map(_norm)].head(20)

lluvia: entidad corregida en 1016 filas | NOMGEO en 168 filas | duplicados restantes 0
temp_min: entidad corregida en 821 filas | NOMGEO en 114 filas | duplicados restantes 0


,idestado,idmunicipio,nombre_entidad,NOMGEO,anio,nomcicloproductivo,lluvia_acumulada_mm,lluvia_anio_anterior_mm,nombre_entidad_orig,NOMGEO_orig
13696,15,111,México,Villa de Allende,2022,OI,123.72,2769.52,Estado de México,Villa de Allende
13684,15,91,México,Teoloyucan,2022,OI,49.51,940.10,Estado de México,Teoloyucan
13687,15,97,México,Texcaltitlán,2022,OI,60.81,874.26,Estado de México,Texcaltitlán
13688,15,99,México,Texcoco,2022,OI,133.88,1471.52,Estado de México,Texcoco
13686,15,96,México,Tequixquiac,2022,OI,32.59,933.55,Estado de México,Tequixquiac
13685,15,95,México,Tepotzotlán,2022,OI,85.30,860.90,Estado de México,Tepotzotlán
13666,15,25,México,Chalco,2022,OI,40.41,663.30,Estado de México,Chalco
13683,15,90,México,Tenango del Valle,2022,OI,125.22,1223.48,Estado de México,Tenango del Valle
13682,15,87,México,Temoaya,2022,OI,222.71,1870.96,Estado de México,Temoaya
4666,15,110,México,Valle de Bravo,2016,OI,276.30,2075.34,Estado de México,Valle de Bravo


In [206]:
# Join de agricolas_agg con lluvias y temp_min (solo anios 2016-2024).
# Ambos lados usan el mismo catalogo CVE -> nombre, asi que el cruce por nombre es exacto.
df_agricolas_agg = df_agricolas_agg[df_agricolas_agg['anio'].isin(ANIOS)].copy()

# homologar los nombres de agricolas_agg al catalogo (idem que se hizo con el clima)
df_agricolas_agg['nomestado'] = (
    df_agricolas_agg['idestado'].map(CAT_ESTADO).fillna(df_agricolas_agg['nomestado'])
)
_mun_agg = df_agricolas_agg.set_index(['idestado', 'idmunicipio']).index.map(CAT_MUNICIPIO)
df_agricolas_agg['nommunicipio'] = (
    pd.Series(_mun_agg, index=df_agricolas_agg.index).fillna(df_agricolas_agg['nommunicipio'])
)

llaves_join = ['idestado', 'idmunicipio', 'nomestado', 'nommunicipio', 'anio', 'nomcicloproductivo']
METRICAS_CLIMA = ['lluvia_acumulada_mm', 'lluvia_anio_anterior_mm',
                  'temp_min_ciclo_min', 'temp_min_anual_anterior']


def _clima_para_join(df):
    d = df.rename(columns={'nombre_entidad': 'nomestado', 'NOMGEO': 'nommunicipio'})
    d = d[d['anio'].isin(ANIOS)]
    cols = llaves_join + [c for c in METRICAS_CLIMA if c in d.columns]
    return d[cols]


df_join = (
    df_agricolas_agg
    .merge(_clima_para_join(df_lluvias), on=llaves_join, how='left')
    .merge(_clima_para_join(df_tmp_min), on=llaves_join, how='left')
)

# Diagnostico de cobertura
n = len(df_join)
con_lluvia = df_join['lluvia_acumulada_mm'].notna().sum()
con_temp = df_join['temp_min_ciclo_min'].notna().sum()
print(f'filas 2016-2024: {n}  (sin multiplicar: {n == len(df_agricolas_agg)})')
print(f'con lluvia   : {con_lluvia} ({con_lluvia / n:.1%})')
print(f'con temp_min : {con_temp} ({con_temp / n:.1%})')

df_join

filas 2016-2024: 40267  (sin multiplicar: True)
con lluvia   : 12764 (31.7%)
con temp_min : 12269 (30.5%)


,anio,idestado,idmunicipio,nomcicloproductivo,nommodalidad,nomestado,nommunicipio,volumenproduccion,cosechada,siniestrada,sembrada,valorproduccion,precio_mean,precio_total,rendimiento_real,tasa_siniestro,lluvia_acumulada_mm,lluvia_anio_anterior_mm,temp_min_ciclo_min,temp_min_anual_anterior
0,2016,1,1,PV,Riego,Aguascalientes,Aguascalientes,2247.15,300.0,0.0,300.0,7805071.04,3473.32,3473.32,7.490500,0.000000,3222.50,5251.71,4.7,1.3
1,2016,1,1,PV,Temporal,Aguascalientes,Aguascalientes,3350.83,5032.0,13.0,5045.0,11255639.02,3359.06,3359.06,0.665904,0.002577,3222.50,5251.71,4.7,1.3
2,2016,1,2,PV,Riego,Aguascalientes,Asientos,3340.00,630.0,0.0,630.0,11690000.00,3500.00,3500.00,5.301587,0.000000,356.70,645.61,5.6,1.4
3,2016,1,2,PV,Temporal,Aguascalientes,Asientos,2770.00,4525.0,0.0,4525.0,9695000.00,3500.00,3500.00,0.612155,0.000000,356.70,645.61,5.6,1.4
4,2016,1,3,PV,Riego,Aguascalientes,Calvillo,333.00,58.0,0.0,58.0,1251800.28,3759.16,3759.16,5.741379,0.000000,2380.77,3200.04,7.8,5.3
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
40262,2024,32,56,PV,Riego,Zacatecas,Zacatecas,6803.40,870.0,0.0,870.0,53645200.00,7550.00,15100.00,7.820000,0.000000,1017.50,560.62,11.2,5.2
40263,2024,32,56,PV,Temporal,Zacatecas,Zacatecas,574.00,700.0,0.0,700.0,3478440.00,6060.00,6060.00,0.820000,0.000000,1017.50,560.62,11.2,5.2
40264,2024,32,57,PV,Riego,Zacatecas,Trancoso,4232.80,520.0,0.0,520.0,32486740.00,7450.00,14900.00,8.140000,0.000000,NaN,NaN,NaN,NaN
40265,2024,32,57,PV,Temporal,Zacatecas,Trancoso,555.75,975.0,0.0,975.0,3396627.29,6111.79,6111.79,0.570000,0.000000,NaN,NaN,NaN,NaN


In [207]:
# Agregar a df_join la informacion de sequia de df_sequia:
#   nivel_sequia_max         -> max. de sequia DURANTE el ciclo (tratamiento)
#   nivel_sequia_prev90_max  -> max. de sequia en los ~90 dias PREVIOS al inicio del ciclo
#                               (condicion antecedente, previa al tratamiento)
# df_sequia ya es internamente consistente (cada CVE_ENT -> 1 ENTIDAD, cada CVE -> 1 NOMBRE_MUN)
# y no trae duplicados por (CVE, anio, ciclo); solo homologamos los nombres al catalogo agricola.
assert 'nivel_sequia_prev90_max' in df_sequia.columns, (
    'Regenera datos_sequia.csv corriendo 01_limpieza_datos_sequia.ipynb'
)
COLS_SEQ = ['nivel_sequia_max', 'nivel_sequia_prev90_max']

_seq = df_sequia.rename(columns={'CVE_ENT': 'idestado', 'CVE_MUN': 'idmunicipio'}).copy()
_seq['idestado'] = _seq['idestado'].astype(int)
_seq['idmunicipio'] = _seq['idmunicipio'].astype(int)
_seq['nomestado'] = _seq['idestado'].map(CAT_ESTADO).fillna(_seq['ENTIDAD'])
_seq['nommunicipio'] = pd.Series(
    _seq.set_index(['idestado', 'idmunicipio']).index.map(CAT_MUNICIPIO), index=_seq.index
).fillna(_seq['NOMBRE_MUN'])

# Los registros sin nivel (NaN del Monitor) se marcan con el centinela -1
for _c in COLS_SEQ:
    _seq[_c] = _seq[_c].fillna(-1).astype(int)

_seq = _seq[_seq['anio'].isin(ANIOS)][llaves_join + COLS_SEQ]

df_join = df_join.drop(columns=COLS_SEQ, errors='ignore').merge(
    _seq, on=llaves_join, how='left'
)

# Diagnostico de cobertura
n = len(df_join)
for _c in COLS_SEQ:
    con = df_join[_c].notna().sum()
    print(f'{_c:24s}: con fila de sequia {con} ({con / n:.1%})')
print(f'filas: {n}  (sin multiplicar: {n == len(df_agricolas_agg)})')
print('\nnivel_sequia_max:')
print(df_join['nivel_sequia_max'].value_counts(dropna=False).sort_index().to_string())
print('\nnivel_sequia_prev90_max:')
print(df_join['nivel_sequia_prev90_max'].value_counts(dropna=False).sort_index().to_string())

df_join

nivel_sequia_max        : con fila de sequia 40267 (100.0%)
nivel_sequia_prev90_max : con fila de sequia 40267 (100.0%)
filas: 40267  (sin multiplicar: True)

nivel_sequia_max:
nivel_sequia_max
-1     4021
 0    12826
 1    12519
 2     6677
 3     3559
 4      665

nivel_sequia_prev90_max:
nivel_sequia_prev90_max
-1    10463
 0    14528
 1     8653
 2     4598
 3     1711
 4      314


,anio,idestado,idmunicipio,nomcicloproductivo,nommodalidad,nomestado,nommunicipio,volumenproduccion,cosechada,siniestrada,sembrada,valorproduccion,precio_mean,precio_total,rendimiento_real,tasa_siniestro,lluvia_acumulada_mm,lluvia_anio_anterior_mm,temp_min_ciclo_min,temp_min_anual_anterior,nivel_sequia_max,nivel_sequia_prev90_max
0,2016,1,1,PV,Riego,Aguascalientes,Aguascalientes,2247.15,300.0,0.0,300.0,7805071.04,3473.32,3473.32,7.490500,0.000000,3222.50,5251.71,4.7,1.3,0,-1
1,2016,1,1,PV,Temporal,Aguascalientes,Aguascalientes,3350.83,5032.0,13.0,5045.0,11255639.02,3359.06,3359.06,0.665904,0.002577,3222.50,5251.71,4.7,1.3,0,-1
2,2016,1,2,PV,Riego,Aguascalientes,Asientos,3340.00,630.0,0.0,630.0,11690000.00,3500.00,3500.00,5.301587,0.000000,356.70,645.61,5.6,1.4,0,-1
3,2016,1,2,PV,Temporal,Aguascalientes,Asientos,2770.00,4525.0,0.0,4525.0,9695000.00,3500.00,3500.00,0.612155,0.000000,356.70,645.61,5.6,1.4,0,-1
4,2016,1,3,PV,Riego,Aguascalientes,Calvillo,333.00,58.0,0.0,58.0,1251800.28,3759.16,3759.16,5.741379,0.000000,2380.77,3200.04,7.8,5.3,0,-1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
40262,2024,32,56,PV,Riego,Zacatecas,Zacatecas,6803.40,870.0,0.0,870.0,53645200.00,7550.00,15100.00,7.820000,0.000000,1017.50,560.62,11.2,5.2,2,3
40263,2024,32,56,PV,Temporal,Zacatecas,Zacatecas,574.00,700.0,0.0,700.0,3478440.00,6060.00,6060.00,0.820000,0.000000,1017.50,560.62,11.2,5.2,2,3
40264,2024,32,57,PV,Riego,Zacatecas,Trancoso,4232.80,520.0,0.0,520.0,32486740.00,7450.00,14900.00,8.140000,0.000000,NaN,NaN,NaN,NaN,1,3
40265,2024,32,57,PV,Temporal,Zacatecas,Trancoso,555.75,975.0,0.0,975.0,3396627.29,6111.79,6111.79,0.570000,0.000000,NaN,NaN,NaN,NaN,1,3


In [208]:
# Agregar a df_join los indicadores de cuenca / acuifero por municipio x AÑO (01_cuencas_acuifero.ipynb).
# Usa el año CONAGUA mas proximo por año del panel -> YA NO es invariante en el tiempo,
# se une por (idestado, idmunicipio, anio).
_ruta_ca = Path('/Users/jaydymarchan/Desktop/causalidad/data/02_processed/cuencas_acuifero_municipio.csv')
assert _ruta_ca.exists(), 'Genera cuencas_acuifero_municipio.csv corriendo 01_cuencas_acuifero.ipynb'

COLS_CA = ['tiene_acuifero_sobreexplotado', 'tiene_cuenca_sin_disp', 'acuifero_disp', 'cuencas_disp',
           'acuifero_condicion_ok']
_ca = (pd.read_csv(_ruta_ca)
       .astype({'idestado': int, 'idmunicipio': int, 'anio': int})
       [['idestado', 'idmunicipio', 'anio'] + COLS_CA]
       .drop_duplicates(['idestado', 'idmunicipio', 'anio']))
assert _ca.duplicated(['idestado', 'idmunicipio', 'anio']).sum() == 0, 'cuencas_acuifero con (municipio, anio) duplicados'

df_join = df_join.drop(columns=COLS_CA, errors='ignore').merge(
    _ca, on=['idestado', 'idmunicipio', 'anio'], how='left'
)

# Diagnostico de cobertura
n = len(df_join)
con_ca = df_join['acuifero_disp'].notna().sum()
print(f'filas: {n}  (sin multiplicar: {n == len(df_agricolas_agg)})')
print(f'con cuenca/acuifero: {con_ca} ({con_ca / n:.1%})')
print('tiene_acuifero_sobreexplotado:', df_join['tiene_acuifero_sobreexplotado'].value_counts(dropna=False).sort_index().to_dict())
print('tiene_cuenca_sin_disp       :', df_join['tiene_cuenca_sin_disp'].value_counts(dropna=False).sort_index().to_dict())

df_join

filas: 40267  (sin multiplicar: True)
con cuenca/acuifero: 40267 (100.0%)
tiene_acuifero_sobreexplotado: {0: 31422, 1: 8845}
tiene_cuenca_sin_disp       : {0: 20544, 1: 19723}


,anio,idestado,idmunicipio,nomcicloproductivo,nommodalidad,nomestado,nommunicipio,volumenproduccion,cosechada,siniestrada,sembrada,valorproduccion,precio_mean,precio_total,rendimiento_real,tasa_siniestro,lluvia_acumulada_mm,lluvia_anio_anterior_mm,temp_min_ciclo_min,temp_min_anual_anterior,nivel_sequia_max,nivel_sequia_prev90_max,tiene_acuifero_sobreexplotado,tiene_cuenca_sin_disp,acuifero_disp,cuencas_disp,acuifero_condicion_ok
0,2016,1,1,PV,Riego,Aguascalientes,Aguascalientes,2247.15,300.0,0.0,300.0,7805071.04,3473.32,3473.32,7.490500,0.000000,3222.50,5251.71,4.7,1.3,0,-1,0,1,-29.64,249.77,0
1,2016,1,1,PV,Temporal,Aguascalientes,Aguascalientes,3350.83,5032.0,13.0,5045.0,11255639.02,3359.06,3359.06,0.665904,0.002577,3222.50,5251.71,4.7,1.3,0,-1,0,1,-29.64,249.77,0
2,2016,1,2,PV,Riego,Aguascalientes,Asientos,3340.00,630.0,0.0,630.0,11690000.00,3500.00,3500.00,5.301587,0.000000,356.70,645.61,5.6,1.4,0,-1,0,1,-29.64,249.77,0
3,2016,1,2,PV,Temporal,Aguascalientes,Asientos,2770.00,4525.0,0.0,4525.0,9695000.00,3500.00,3500.00,0.612155,0.000000,356.70,645.61,5.6,1.4,0,-1,0,1,-29.64,249.77,0
4,2016,1,3,PV,Riego,Aguascalientes,Calvillo,333.00,58.0,0.0,58.0,1251800.28,3759.16,3759.16,5.741379,0.000000,2380.77,3200.04,7.8,5.3,0,-1,0,1,-29.64,249.77,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
40262,2024,32,56,PV,Riego,Zacatecas,Zacatecas,6803.40,870.0,0.0,870.0,53645200.00,7550.00,15100.00,7.820000,0.000000,1017.50,560.62,11.2,5.2,2,3,1,1,-10.38,95.20,1
40263,2024,32,56,PV,Temporal,Zacatecas,Zacatecas,574.00,700.0,0.0,700.0,3478440.00,6060.00,6060.00,0.820000,0.000000,1017.50,560.62,11.2,5.2,2,3,1,1,-10.38,95.20,1
40264,2024,32,57,PV,Riego,Zacatecas,Trancoso,4232.80,520.0,0.0,520.0,32486740.00,7450.00,14900.00,8.140000,0.000000,NaN,NaN,NaN,NaN,1,3,1,0,-10.38,21.04,1
40265,2024,32,57,PV,Temporal,Zacatecas,Trancoso,555.75,975.0,0.0,975.0,3396627.29,6111.79,6111.79,0.570000,0.000000,NaN,NaN,NaN,NaN,1,3,1,0,-10.38,21.04,1


In [209]:
df_join

,anio,idestado,idmunicipio,nomcicloproductivo,nommodalidad,nomestado,nommunicipio,volumenproduccion,cosechada,siniestrada,sembrada,valorproduccion,precio_mean,precio_total,rendimiento_real,tasa_siniestro,lluvia_acumulada_mm,lluvia_anio_anterior_mm,temp_min_ciclo_min,temp_min_anual_anterior,nivel_sequia_max,nivel_sequia_prev90_max,tiene_acuifero_sobreexplotado,tiene_cuenca_sin_disp,acuifero_disp,cuencas_disp,acuifero_condicion_ok
0,2016,1,1,PV,Riego,Aguascalientes,Aguascalientes,2247.15,300.0,0.0,300.0,7805071.04,3473.32,3473.32,7.490500,0.000000,3222.50,5251.71,4.7,1.3,0,-1,0,1,-29.64,249.77,0
1,2016,1,1,PV,Temporal,Aguascalientes,Aguascalientes,3350.83,5032.0,13.0,5045.0,11255639.02,3359.06,3359.06,0.665904,0.002577,3222.50,5251.71,4.7,1.3,0,-1,0,1,-29.64,249.77,0
2,2016,1,2,PV,Riego,Aguascalientes,Asientos,3340.00,630.0,0.0,630.0,11690000.00,3500.00,3500.00,5.301587,0.000000,356.70,645.61,5.6,1.4,0,-1,0,1,-29.64,249.77,0
3,2016,1,2,PV,Temporal,Aguascalientes,Asientos,2770.00,4525.0,0.0,4525.0,9695000.00,3500.00,3500.00,0.612155,0.000000,356.70,645.61,5.6,1.4,0,-1,0,1,-29.64,249.77,0
4,2016,1,3,PV,Riego,Aguascalientes,Calvillo,333.00,58.0,0.0,58.0,1251800.28,3759.16,3759.16,5.741379,0.000000,2380.77,3200.04,7.8,5.3,0,-1,0,1,-29.64,249.77,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
40262,2024,32,56,PV,Riego,Zacatecas,Zacatecas,6803.40,870.0,0.0,870.0,53645200.00,7550.00,15100.00,7.820000,0.000000,1017.50,560.62,11.2,5.2,2,3,1,1,-10.38,95.20,1
40263,2024,32,56,PV,Temporal,Zacatecas,Zacatecas,574.00,700.0,0.0,700.0,3478440.00,6060.00,6060.00,0.820000,0.000000,1017.50,560.62,11.2,5.2,2,3,1,1,-10.38,95.20,1
40264,2024,32,57,PV,Riego,Zacatecas,Trancoso,4232.80,520.0,0.0,520.0,32486740.00,7450.00,14900.00,8.140000,0.000000,NaN,NaN,NaN,NaN,1,3,1,0,-10.38,21.04,1
40265,2024,32,57,PV,Temporal,Zacatecas,Trancoso,555.75,975.0,0.0,975.0,3396627.29,6111.79,6111.79,0.570000,0.000000,NaN,NaN,NaN,NaN,1,3,1,0,-10.38,21.04,1


In [210]:
# ¿Todos los municipios/anios de df_sequia tienen registro agricola?  -> NO
_sq = df_sequia.rename(columns={'CVE_ENT': 'idestado', 'CVE_MUN': 'idmunicipio'}).copy()
_sq['idestado'] = _sq['idestado'].astype(int)
_sq['idmunicipio'] = _sq['idmunicipio'].astype(int)

_ag_mun = set(map(tuple, df_agricolas[['idestado', 'idmunicipio']].drop_duplicates().values))
_sq_mun = set(map(tuple, _sq[['idestado', 'idmunicipio']].drop_duplicates().values))

_sq16 = _sq[_sq['anio'].isin(ANIOS)]
_ag16 = df_agricolas[df_agricolas['anio'].isin(ANIOS)]
_k = ['idestado', 'idmunicipio', 'anio', 'nomcicloproductivo']
_sq_keys = set(map(tuple, _sq16[_k].drop_duplicates().values))
_ag_keys = set(map(tuple, _ag16[_k].drop_duplicates().values))

print(f'municipios en df_sequia: {len(_sq_mun)}')
print(f'  sin NINGUN registro agricola (ningun anio): {len(_sq_mun - _ag_mun)}')
print(f'combinaciones (municipio, anio, ciclo) en df_sequia 2016-2024: {len(_sq_keys)}')
print(f'  sin registro agricola: {len(_sq_keys - _ag_keys)} ({len(_sq_keys - _ag_keys) / len(_sq_keys):.1%})')

# municipios de sequia que nunca aparecen en agricola
_nm = _sq.groupby(['idestado', 'idmunicipio'])['NOMBRE_MUN'].first()
_en = _sq.groupby(['idestado', 'idmunicipio'])['ENTIDAD'].first()
df_sequia_sin_agricola = (
    pd.DataFrame(sorted(_sq_mun - _ag_mun), columns=['idestado', 'idmunicipio'])
    .assign(nomestado=lambda d: [_en[tuple(x)] for x in d[['idestado', 'idmunicipio']].values],
            nommunicipio=lambda d: [_nm[tuple(x)] for x in d[['idestado', 'idmunicipio']].values],
            CVE=lambda d: d['idestado'].astype(str).str.zfill(2) + d['idmunicipio'].astype(str).str.zfill(3))
    [['CVE', 'idestado', 'idmunicipio', 'nomestado', 'nommunicipio']]
)
df_sequia_sin_agricola

municipios en df_sequia: 2478
  sin NINGUN registro agricola (ningun anio): 75
combinaciones (municipio, anio, ciclo) en df_sequia 2016-2024: 44604
  sin registro agricola: 14031 (31.5%)


,CVE,idestado,idmunicipio,nomestado,nommunicipio
0,02005,2,5,Baja California,Playas de Rosarito
1,02007,2,7,Baja California,San Felipe**
2,08004,8,4,Chihuahua,Aquiles Serdán
3,08015,8,15,Chihuahua,Coyame del Sotol
4,08021,8,21,Chihuahua,Delicias
...,...,...,...,...,...
70,28027,28,27,Tamaulipas,Nuevo Laredo
71,28038,28,38,Tamaulipas,Tampico
72,30028,30,28,Veracruz de Ignacio de la Llave,Boca del Río
73,30118,30,118,Veracruz de Ignacio de la Llave,Orizaba


In [211]:
lista_municpio_total = df_agricolas_agg[df_agricolas_agg['nommunicipio'].notnull()]['nommunicipio'].tolist()

# lista_municpio_total.unique
# len(set(lista_municpio_total)) # 2257


In [212]:
# Municipios / anio / ciclo SIN informacion de lluvia NI de temp_min
_sin_clima = (
    df_join['lluvia_acumulada_mm'].isna()
    & df_join['temp_min_ciclo_min'].isna()
)

df_sin_clima = (
    df_join.loc[_sin_clima,
                ['anio', 'idestado', 'idmunicipio', 'nomestado', 'nommunicipio', 'nomcicloproductivo']]
    .drop_duplicates()
    .sort_values(['anio', 'idestado', 'idmunicipio', 'nomcicloproductivo'])
    .reset_index(drop=True)
)

print(f'{len(df_sin_clima)} de {len(df_join)} filas ({len(df_sin_clima) / len(df_join):.1%}) sin lluvia ni temp_min')
print(f'municipios distintos afectados: {df_sin_clima[["idestado", "idmunicipio"]].drop_duplicates().shape[0]}')

df_sin_clima.to_csv(
    '/Users/jaydymarchan/Desktop/causalidad/data/02_processed/municipios_sin_clima.csv',
    index=False, encoding='utf-8',
)

df_sin_clima

19973 de 40267 filas (49.6%) sin lluvia ni temp_min
municipios distintos afectados: 1745


,anio,idestado,idmunicipio,nomestado,nommunicipio,nomcicloproductivo
0,2016,1,11,Aguascalientes,San Francisco de Los Romo,PV
1,2016,4,1,Campeche,Calkiní,OI
2,2016,4,1,Campeche,Calkiní,PV
3,2016,4,8,Campeche,Tenabo,OI
4,2016,4,8,Campeche,Tenabo,PV
...,...,...,...,...,...,...
19968,2024,32,41,Zacatecas,El Salvador,PV
19969,2024,32,50,Zacatecas,Vetagrande,PV
19970,2024,32,52,Zacatecas,Villa García,PV
19971,2024,32,57,Zacatecas,Trancoso,PV


In [213]:
df_sequia = pd.read_csv('/Users/jaydymarchan/Desktop/causalidad/data/02_processed/datos_sequia.csv', encoding='utf-8')
print(df_sequia.shape[0]) # 59472
df_sequia.head()

59472


,CVE_CONCATENADA,CVE_ENT,CVE_MUN,NOMBRE_MUN,ENTIDAD,ORG_CUENCA*,CLV_OC,CON_CUENCA,CVE_CONC,anio,nomcicloproductivo,nivel_sequia_max,nivel_sequia_prev90_max
0,1001,1,1,Aguascalientes,Aguascalientes,Lerma-Santiago-Pacífico,VIII,Rio Santiago,16,2013,OI,NaN,NaN
1,1002,1,2,Asientos,Aguascalientes,Lerma-Santiago-Pacífico,VIII,Rio Santiago,16,2013,OI,NaN,NaN
2,1003,1,3,Calvillo,Aguascalientes,Lerma-Santiago-Pacífico,VIII,Rio Santiago,16,2013,OI,NaN,NaN
3,1004,1,4,Cosío,Aguascalientes,Lerma-Santiago-Pacífico,VIII,Rio Santiago,16,2013,OI,NaN,NaN
4,1005,1,5,Jesús María,Aguascalientes,Lerma-Santiago-Pacífico,VIII,Rio Santiago,16,2013,OI,NaN,NaN


In [214]:
# ¿Cuales de los municipios sin lluvia son municipios NUEVOS (creados despues de 2016)?
# Criterio: su CVE (idestado, idmunicipio) no aparece en ningun cierre agricola 2013-2016.
_RAW = '/Users/jaydymarchan/Desktop/causalidad/data/01_raw/datos_agricolas/Cierre_agricola_mun_{}.csv'

_cve_por_anio = {}
for _y in range(2013, 2025):
    _d = pd.read_csv(_RAW.format(_y), encoding='latin-1', low_memory=False)
    _d.columns = [c.lower() for c in _d.columns]
    _ie = next(c for c in _d.columns if 'idestado' in c)
    _im = next(c for c in _d.columns if 'idmun' in c)
    _cve_por_anio[_y] = set(map(tuple, _d[[_ie, _im]].dropna().astype(int).drop_duplicates().values))

_universo_pre2017 = set().union(*(_cve_por_anio[y] for y in range(2013, 2017)))


def _primer_anio(estado, municipio):
    return min((y for y, s in _cve_por_anio.items() if (estado, municipio) in s), default=pd.NA)


_muni_sin_clima = df_sin_clima[['idestado', 'idmunicipio', 'nomestado', 'nommunicipio']].drop_duplicates()
_muni_sin_clima['es_nuevo_post2016'] = ~_muni_sin_clima.apply(
    lambda r: (r['idestado'], r['idmunicipio']) in _universo_pre2017, axis=1
)
_muni_sin_clima['primer_anio_agricola'] = _muni_sin_clima.apply(
    lambda r: _primer_anio(r['idestado'], r['idmunicipio']), axis=1
)

df_sin_clima_nuevos = (
    _muni_sin_clima[_muni_sin_clima['es_nuevo_post2016']]
    .assign(CVE=lambda d: d['idestado'].astype(str).str.zfill(2) + d['idmunicipio'].astype(str).str.zfill(3))
    [['CVE', 'idestado', 'idmunicipio', 'nomestado', 'nommunicipio', 'primer_anio_agricola']]
    .sort_values(['idestado', 'idmunicipio'])
    .reset_index(drop=True)
)

print(f'{len(_muni_sin_clima)} municipios sin lluvia; de ellos {len(df_sin_clima_nuevos)} son nuevos (post-2016)')
df_sin_clima_nuevos

1745 municipios sin lluvia; de ellos 7 son nuevos (post-2016)


,CVE,idestado,idmunicipio,nomestado,nommunicipio,primer_anio_agricola
0,04012,4,12,Campeche,Seybaplaya,2022
1,04013,4,13,Campeche,Dzitbalché,2022
2,07120,7,120,Chiapas,Capitán Luis Ángel Vidal,2021
3,07121,7,121,Chiapas,Rincón Chamula San Pedro,2021
4,07122,7,122,Chiapas,El Parral,2021
5,07125,7,125,Chiapas,Honduras de la Sierra,2022
6,17034,17,34,Morelos,Coatetelco,2021


In [215]:
# Municipios que NO tuvieron registro agricola en todos los anios del periodo 2016-2024
_ag_2016_2024 = df_agricolas[df_agricolas['anio'].between(2016, 2024)]

_cob = (
    _ag_2016_2024.groupby(['idestado', 'idmunicipio'])
    .agg(nomestado=('nomestado', 'first'),
         nommunicipio=('nommunicipio', 'first'),
         n_anios=('anio', 'nunique'),
         anios=('anio', lambda s: sorted(s.unique())))
    .reset_index()
)
_cob['anios_faltantes'] = _cob['anios'].apply(lambda a: [y for y in range(2016, 2025) if y not in a])

df_cobertura_incompleta = (
    _cob[_cob['n_anios'] < 9]
    .drop(columns='anios')
    .sort_values(['n_anios', 'idestado', 'idmunicipio'])
    .reset_index(drop=True)
)

print(f'municipios con registro agricola 2016-2024: {len(_cob)}')
print(f'  - con los 9 anios completos : {(_cob["n_anios"] == 9).sum()}')
print(f'  - con cobertura incompleta  : {len(df_cobertura_incompleta)}')
print('\npor estado:')
print(df_cobertura_incompleta['nomestado'].value_counts().to_string())
df_cobertura_incompleta

municipios con registro agricola 2016-2024: 2386
  - con los 9 anios completos : 2262
  - con cobertura incompleta  : 124

por estado:
nomestado
Nuevo León         33
Sonora             32
Chihuahua          13
Coahuila            8
Yucatán             8
Chiapas             6
Tamaulipas          5
Baja California     4
Morelos             3
Campeche            2
Durango             2
Jalisco             2
Sinaloa             2
Puebla              1
México              1
Nayarit             1
Oaxaca              1


,idestado,idmunicipio,nomestado,nommunicipio,n_anios,anios_faltantes
0,2,3,Baja California,Tecate,1,"[2016, 2018, 2019, 2020, 2021, 2022, 2023, 2024]"
1,8,16,Chihuahua,La Cruz,1,"[2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024]"
2,8,24,Chihuahua,Santa Isabel,1,"[2016, 2017, 2018, 2019, 2020, 2022, 2023, 2024]"
3,8,59,Chihuahua,San Francisco del Oro,1,"[2016, 2017, 2019, 2020, 2021, 2022, 2023, 2024]"
4,8,62,Chihuahua,Saucillo,1,"[2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024]"
...,...,...,...,...,...,...
119,31,39,Yucatán,Ixil,8,[2016]
120,31,41,Yucatán,Kanasín,8,[2016]
121,31,51,Yucatán,Mocochá,8,[2016]
122,31,59,Yucatán,Progreso,8,[2016]


In [216]:
df_sin_clima.groupby([ 'nomestado','nommunicipio']).size().reset_index(name='n').sort_values('n', ascending=False)

,nomestado,nommunicipio,n
785,Oaxaca,San Juan Mixtepec,27
887,Oaxaca,San Pedro Mixtepec,20
847,Oaxaca,San Miguel Peras,18
1101,Oaxaca,Teotitlán de Flores Magón,18
810,Oaxaca,San Lucas Camotlán,18
...,...,...,...
159,Durango,Tepehuanes,1
1369,Sonora,Divisaderos,1
1374,Sonora,Magdalena,1
71,Chiapas,Siltepec,1


In [217]:
# 1743 

In [218]:
df_lluvias.head()

,idestado,idmunicipio,nombre_entidad,NOMGEO,anio,nomcicloproductivo,lluvia_acumulada_mm,lluvia_anio_anterior_mm,nombre_entidad_orig,NOMGEO_orig
0,1,1,Aguascalientes,Aguascalientes,2013,OI,238.34,NaN,Aguascalientes,Aguascalientes
11827,32,14,Zacatecas,General Francisco R. Murguía,2020,PV,205.38,179.57,Zacatecas,General Francisco R. Murguía
11834,32,22,Zacatecas,Juan Aldama,2020,PV,572.34,184.72,Zacatecas,Juan Aldama
11833,32,20,Zacatecas,Jerez,2020,PV,731.19,787.45,Zacatecas,Jerez
11832,32,19,Zacatecas,Jalpa,2020,PV,386.21,523.22,Zacatecas,Jalpa


In [219]:
# df_sequia.head()

In [220]:
df_tmp_min.head()

,idestado,idmunicipio,nombre_entidad,NOMGEO,anio,nomcicloproductivo,temp_min_ciclo_min,temp_min_anual_anterior,nombre_entidad_orig,NOMGEO_orig
0,1,1,Aguascalientes,Aguascalientes,2013,OI,0.2,NaN,Aguascalientes,Aguascalientes
10617,30,2,Veracruz,Acatlán,2021,OI,6.4,6.400000,Veracruz,Acatlán
10619,30,4,Veracruz,Actopan,2021,OI,18.1,19.293103,Veracruz,Actopan
10620,30,10,Veracruz,Altotonga,2021,OI,5.3,5.300000,Veracruz,Altotonga
10621,30,11,Veracruz,Alvarado,2021,OI,17.6,17.600000,Veracruz,Alvarado


In [221]:
df_join.shape

(40267, 27)

In [222]:
df_join.head()

,anio,idestado,idmunicipio,nomcicloproductivo,nommodalidad,nomestado,nommunicipio,volumenproduccion,cosechada,siniestrada,sembrada,valorproduccion,precio_mean,precio_total,rendimiento_real,tasa_siniestro,lluvia_acumulada_mm,lluvia_anio_anterior_mm,temp_min_ciclo_min,temp_min_anual_anterior,nivel_sequia_max,nivel_sequia_prev90_max,tiene_acuifero_sobreexplotado,tiene_cuenca_sin_disp,acuifero_disp,cuencas_disp,acuifero_condicion_ok
0,2016,1,1,PV,Riego,Aguascalientes,Aguascalientes,2247.15,300.0,0.0,300.0,7805071.04,3473.32,3473.32,7.490500,0.000000,3222.50,5251.71,4.7,1.3,0,-1,0,1,-29.64,249.77,0
1,2016,1,1,PV,Temporal,Aguascalientes,Aguascalientes,3350.83,5032.0,13.0,5045.0,11255639.02,3359.06,3359.06,0.665904,0.002577,3222.50,5251.71,4.7,1.3,0,-1,0,1,-29.64,249.77,0
2,2016,1,2,PV,Riego,Aguascalientes,Asientos,3340.00,630.0,0.0,630.0,11690000.00,3500.00,3500.00,5.301587,0.000000,356.70,645.61,5.6,1.4,0,-1,0,1,-29.64,249.77,0
3,2016,1,2,PV,Temporal,Aguascalientes,Asientos,2770.00,4525.0,0.0,4525.0,9695000.00,3500.00,3500.00,0.612155,0.000000,356.70,645.61,5.6,1.4,0,-1,0,1,-29.64,249.77,0
4,2016,1,3,PV,Riego,Aguascalientes,Calvillo,333.00,58.0,0.0,58.0,1251800.28,3759.16,3759.16,5.741379,0.000000,2380.77,3200.04,7.8,5.3,0,-1,0,1,-29.64,249.77,0


In [223]:
# =========================================================
# CONTEOS DE COBERTURA  (periodo 2016-2024)
# =========================================================
MUN = ['idestado', 'idmunicipio']
ANIOS_2016_2024 = list(range(2016, 2025))

# df_join ya esta acotado a 2016-2024; aun asi se filtra explicito por claridad
_dj = df_join[df_join['anio'].isin(ANIOS_2016_2024)]

# --- 1 y 2) Municipios / municipios-anio con nulos en nivel_sequia_max ----
# En df_join los faltantes de sequia se marcaron con el centinela -1 (ver celda del merge);
# se consideran "sin dato" tanto NaN como -1.
sin_sequia = _dj['nivel_sequia_max'].isna()

n_mun_sin_sequia      = _dj.loc[sin_sequia, MUN].drop_duplicates().shape[0]
n_mun_anio_sin_sequia = _dj.loc[sin_sequia, MUN + ['anio']].drop_duplicates().shape[0]

print('NIVEL_SEQUIA_MAX sin dato (NaN o -1)  |  2016-2024')
print(f'  filas                  : {int(sin_sequia.sum())}')
print(f'  municipios unicos      : {n_mun_sin_sequia}')
print(f'  municipios-anio unicos : {n_mun_anio_sin_sequia}')

# --- 3 y 4) Municipios / municipios-anio SIN lluvia_acumulada_mm NI temp_min_ciclo_min
sin_clima = _dj['lluvia_acumulada_mm'].isna() & _dj['temp_min_ciclo_min'].isna()

n_mun_sin_clima      = _dj.loc[sin_clima, MUN].drop_duplicates().shape[0]
n_mun_anio_sin_clima = _dj.loc[sin_clima, MUN + ['anio']].drop_duplicates().shape[0]

print('\nSIN lluvia_acumulada_mm Y SIN temp_min_ciclo_min  |  2016-2024')
print(f'  filas                  : {int(sin_clima.sum())}')
print(f'  municipios unicos      : {n_mun_sin_clima}')
print(f'  municipios-anio unicos : {n_mun_anio_sin_clima}')

# --- 5) Agrupado de municipios: numero de anios / registros en datos agricolas 2016-2024
_ag = df_agricolas[df_agricolas['anio'].isin(ANIOS_2016_2024)]

df_registros_agricolas_por_mun = (
    _ag
    .groupby(MUN + ['nomestado', 'nommunicipio'], as_index=False)
    .agg(n_anios=('anio', 'nunique'),
         n_registros=('anio', 'size'),
         anio_min=('anio', 'min'),
         anio_max=('anio', 'max'),
         anios=('anio', lambda s: sorted(s.unique())))
    .sort_values(['n_anios', 'idestado', 'idmunicipio'], ascending=[False, True, True])
    .reset_index(drop=True)
)

print(f'\nMunicipios con datos agricolas 2016-2024: {len(df_registros_agricolas_por_mun)}')
print('distribucion por numero de anios con registro (de 9 posibles):')
print(df_registros_agricolas_por_mun['n_anios'].value_counts().sort_index().to_string())
display(df_registros_agricolas_por_mun)

NIVEL_SEQUIA_MAX sin dato (NaN o -1)  |  2016-2024
  filas                  : 0
  municipios unicos      : 0
  municipios-anio unicos : 0

SIN lluvia_acumulada_mm Y SIN temp_min_ciclo_min  |  2016-2024
  filas                  : 25763
  municipios unicos      : 1745
  municipios-anio unicos : 13906

Municipios con datos agricolas 2016-2024: 2386
distribucion por numero de anios con registro (de 9 posibles):
n_anios
1      16
2       6
3      17
4      13
5      15
6      12
7       8
8      37
9    2262


,idestado,idmunicipio,nomestado,nommunicipio,n_anios,n_registros,anio_min,anio_max,anios
0,1,1,Aguascalientes,Aguascalientes,9,18,2016,2024,"[2016, 2017, 2018, 2019, 2020, 2021, 2022, 202..."
1,1,2,Aguascalientes,Asientos,9,18,2016,2024,"[2016, 2017, 2018, 2019, 2020, 2021, 2022, 202..."
2,1,3,Aguascalientes,Calvillo,9,18,2016,2024,"[2016, 2017, 2018, 2019, 2020, 2021, 2022, 202..."
3,1,4,Aguascalientes,Cosío,9,18,2016,2024,"[2016, 2017, 2018, 2019, 2020, 2021, 2022, 202..."
4,1,5,Aguascalientes,Jesús María,9,18,2016,2024,"[2016, 2017, 2018, 2019, 2020, 2021, 2022, 202..."
...,...,...,...,...,...,...,...,...,...
2381,26,17,Sonora,Caborca,1,1,2021,2021,[2021]
2382,26,19,Sonora,Cananea,1,1,2024,2024,[2024]
2383,26,24,Sonora,Divisaderos,1,2,2024,2024,[2024]
2384,26,36,Sonora,Magdalena,1,1,2016,2016,[2016]


In [224]:
# =========================================================
# TABLA DE MODELADO
#   resultado   : datos agrícolas agregados (municipio × año × ciclo × modalidad, 2016-2024)
#   tratamiento : nivel_sequia_max (Monitor de Sequía)
#                 + nivel_sequia_prev90_max (sequía en los ~90 días previos al ciclo)
#   confusores  : temperatura Tier 1 de Daymet (01_confusores.ipynb)
#                 + altitud media municipal      (01_altura_promedio.ipynb)
#                 + estrés hídrico cuenca/acuífero (01_cuencas_acuifero.ipynb)
#   nota: el grano incluye 'nommodalidad' (Riego/Temporal), que viene de df_agricolas;
#         sequía, clima y confusores municipales se replican en las dos filas del municipio-ciclo.
# =========================================================
RUTA_PROC = Path('/Users/jaydymarchan/Desktop/causalidad/data/02_processed')
LLAVES    = ['idestado', 'idmunicipio', 'anio', 'nomcicloproductivo']
MUNI      = ['idestado', 'idmunicipio']

# --- 1. Resultado: base agrícola (ya viene agregada y acotada a 2016-2024 en la celda del groupby)
agri = df_agricolas_agg[df_agricolas_agg['anio'].between(2016, 2024)].copy()
agri['idestado']    = agri['idestado'].astype(int)
agri['idmunicipio'] = agri['idmunicipio'].astype(int)
assert agri.duplicated(LLAVES + ['nommodalidad']).sum() == 0, 'agri: (llaves, modalidad) duplicadas'

# --- 2. Tratamiento: sequía (df_sequia crudo, conserva los NaN del Monitor)
assert 'nivel_sequia_prev90_max' in df_sequia.columns, (
    'Regenera datos_sequia.csv corriendo 01_limpieza_datos_sequia.ipynb'
)
COLS_SEQ = ['nivel_sequia_max', 'nivel_sequia_prev90_max']   # DURANTE el ciclo / ~90 días previos
seq = (df_sequia
       .rename(columns={'CVE_ENT': 'idestado', 'CVE_MUN': 'idmunicipio'})
       .astype({'idestado': int, 'idmunicipio': int})
       .query('2016 <= anio <= 2024')
       [LLAVES + COLS_SEQ]
       .drop_duplicates(LLAVES))

# --- 3. Confusores: temperatura Daymet Tier 1  (incluye las variables prev90 nuevas)
ruta_conf = RUTA_PROC / 'confusores_temp.csv'
assert ruta_conf.exists(), 'Genera confusores_temp.csv corriendo 01_confusores.ipynb'
conf = pd.read_csv(ruta_conf).astype({'idestado': int, 'idmunicipio': int})
assert conf.duplicated(LLAVES).sum() == 0, 'conf tiene llaves duplicadas'
_need_conf = {'tmean_prev90', 'tmean_prev90_normal', 'tmean_prev90_anom', 'prev90_incompleta'}
assert _need_conf.issubset(conf.columns), (
    f'Regenera confusores_temp.csv (01_confusores.ipynb): faltan {sorted(_need_conf - set(conf.columns))}'
)
COLS_CONF = [c for c in conf.columns if c not in LLAVES]   # se unen TODAS las columnas de confusores

# --- 4. Confusor topográfico: altitud media municipal (invariante en el tiempo -> se une por municipio)
ruta_alt = RUTA_PROC / 'altura_municipios.csv'
assert ruta_alt.exists(), 'Genera altura_municipios.csv corriendo 01_altura_promedio.ipynb'
alt = (pd.read_csv(ruta_alt)
       .astype({'idestado': int, 'idmunicipio': int})
       [MUNI + ['altitud_media_m', 'altitud_min_m', 'altitud_max_m', 'altitud_std_m']]
       .drop_duplicates(MUNI))
assert alt.duplicated(MUNI).sum() == 0, 'alt tiene municipios duplicados'
COLS_ALT = [c for c in alt.columns if c not in MUNI]

# --- 5. Estrés hídrico: cuenca / acuífero por municipio x AÑO (01_cuencas_acuifero.ipynb usa el
#        año CONAGUA más próximo por año del panel -> ya NO es invariante en el tiempo, se une por
#        municipio + año)
MUNI_ANIO = MUNI + ['anio']
ruta_ca = RUTA_PROC / 'cuencas_acuifero_municipio.csv'
assert ruta_ca.exists(), 'Genera cuencas_acuifero_municipio.csv corriendo 01_cuencas_acuifero.ipynb'
ca = (pd.read_csv(ruta_ca)
      .astype({'idestado': int, 'idmunicipio': int, 'anio': int})
      [MUNI_ANIO + ['tiene_acuifero_sobreexplotado', 'tiene_cuenca_sin_disp', 'acuifero_disp',
                    'cuencas_disp', 'acuifero_condicion_ok']]
      .drop_duplicates(MUNI_ANIO))
assert ca.duplicated(MUNI_ANIO).sum() == 0, 'cuencas_acuifero con (municipio, año) duplicados'
COLS_CA = [c for c in ca.columns if c not in MUNI_ANIO]

# --- Join: left sobre la base agrícola (no multiplica filas) -----------------
df_modelo = (
    agri
    .merge(seq,  on=LLAVES,     how='left')
    .merge(conf, on=LLAVES,     how='left')
    .merge(alt,  on=MUNI,       how='left')
    .merge(ca,   on=MUNI_ANIO,  how='left')
)
assert len(df_modelo) == len(agri), 'el join multiplicó filas'

# nivel_sequia_max faltante  ->  0  (municipio-ciclo no clasificado en sequía por el Monitor;
# supuesto estándar del producto. Verificar contra MunicipiosSequia.xlsx; el indicador permite revertirlo.)
df_modelo['sequia_no_clasificada']        = df_modelo['nivel_sequia_max'].isna()
df_modelo['sequia_prev90_no_clasificada'] = df_modelo['nivel_sequia_prev90_max'].isna()
df_modelo['nivel_sequia_max']        = df_modelo['nivel_sequia_max'].fillna(0).astype(int)
df_modelo['nivel_sequia_prev90_max'] = df_modelo['nivel_sequia_prev90_max'].fillna(0).astype(int)

# indicadores de fila apta por bloque de confusores
df_modelo['clima_ok']   = df_modelo['ventana_incompleta'].eq(False)          # ventana del ciclo completa en Daymet
df_modelo['prev90_ok']  = df_modelo['prev90_incompleta'].eq(False)           # 90 días previos completos en Daymet
df_modelo['altitud_ok'] = df_modelo['altitud_media_m'].notna()               # municipio con altitud calculada
df_modelo['hidro_ok']   = df_modelo['acuifero_disp'].notna()                 # municipio-año con cuenca/acuífero

# Punto de corte de Y: se descarta la superficie con siniestro MAYORITARIO
# (siniestrada/sembrada >= 50%) -> el resultado en esas filas se apoya en menos de la mitad
# de la superficie sembrada original.
UMBRAL_SINIESTRO = 0.50   # corte fijo (antes: percentil 95 ~ 0.20)
df_modelo['siniestro_ok'] = df_modelo['tasa_siniestro'] < UMBRAL_SINIESTRO   # False = siniestro >= 50%

print(f'df_modelo: {df_modelo.shape}')
print(f'  sequía imputada a 0 (sin clasificar): {df_modelo["sequia_no_clasificada"].sum()}  | prev90 sin clasificar: {df_modelo["sequia_prev90_no_clasificada"].sum()}')
print(f'  con confusores Daymet (clima_ok)    : {df_modelo["clima_ok"].sum()}  '
      f'({df_modelo["clima_ok"].mean():.1%})')
print(f'  con prev90 completo (prev90_ok)     : {df_modelo["prev90_ok"].sum()}  '
      f'({df_modelo["prev90_ok"].mean():.1%})')
print(f'  con altitud municipal (altitud_ok) : {df_modelo["altitud_ok"].sum()}  '
      f'({df_modelo["altitud_ok"].mean():.1%})')
print(f'  con cuenca/acuífero (hidro_ok)     : {df_modelo["hidro_ok"].sum()}  '
      f'({df_modelo["hidro_ok"].mean():.1%})')
print(f'  columnas de confusores unidas      : {COLS_CONF}')
print(f'  columnas de altitud unidas         : {COLS_ALT}')
print(f'  columnas de cuenca/acuífero unidas : {COLS_CA}')
print(f'  filas con acuifero_condicion_ok=0 ("Condición" no publicada esa fuente, ver 01_cuencas_acuifero): '
      f'{int((df_modelo["acuifero_condicion_ok"] == 0).sum())}')
print(f'  modalidad (nommodalidad)           : {df_modelo["nommodalidad"].value_counts().to_dict()}')
print(f'  umbral tasa_siniestro (fijo): {UMBRAL_SINIESTRO:.3f}  '
      f'-> siniestro_ok=False en {int((~df_modelo["siniestro_ok"]).sum())} filas '
      f'({(~df_modelo["siniestro_ok"]).mean():.1%})')
df_modelo.head()

df_modelo: (40267, 56)
  sequía imputada a 0 (sin clasificar): 4021  | prev90 sin clasificar: 10463
  con confusores Daymet (clima_ok)    : 40172  (99.8%)
  con prev90 completo (prev90_ok)     : 40172  (99.8%)
  con altitud municipal (altitud_ok) : 40267  (100.0%)
  con cuenca/acuífero (hidro_ok)     : 40267  (100.0%)
  columnas de confusores unidas      : ['tmean_ciclo', 'tmin_ciclo_mean', 'tmax_ciclo_mean', 'dias_helada', 'gdd_ciclo', 'tmean_inicio', 'tmean_preseason', 'tmean_prev90', 'tmean_prev90_normal', 'tmean_prev90_anom', 'tmean_normal', 'tmean_ciclo_anom', 'tmean_ciclo_anom_z', 'gdd_normal', 'gdd_anom', 'tmean_inicio_anom', 'tmean_preseason_anom', 'n_dias_ventana', 'n_dias_con_dato', 'n_dias_prev90', 'ventana_incompleta', 'prev90_incompleta']
  columnas de altitud unidas         : ['altitud_media_m', 'altitud_min_m', 'altitud_max_m', 'altitud_std_m']
  columnas de cuenca/acuífero unidas : ['tiene_acuifero_sobreexplotado', 'tiene_cuenca_sin_disp', 'acuifero_disp', 'cuencas_disp

,anio,idestado,idmunicipio,nomcicloproductivo,nommodalidad,nomestado,nommunicipio,volumenproduccion,cosechada,siniestrada,sembrada,valorproduccion,precio_mean,precio_total,rendimiento_real,tasa_siniestro,nivel_sequia_max,nivel_sequia_prev90_max,tmean_ciclo,tmin_ciclo_mean,tmax_ciclo_mean,dias_helada,gdd_ciclo,tmean_inicio,tmean_preseason,tmean_prev90,tmean_prev90_normal,tmean_prev90_anom,tmean_normal,tmean_ciclo_anom,tmean_ciclo_anom_z,gdd_normal,gdd_anom,tmean_inicio_anom,tmean_preseason_anom,n_dias_ventana,n_dias_con_dato,n_dias_prev90,ventana_incompleta,prev90_incompleta,altitud_media_m,altitud_min_m,altitud_max_m,altitud_std_m,tiene_acuifero_sobreexplotado,tiene_cuenca_sin_disp,acuifero_disp,cuencas_disp,acuifero_condicion_ok,sequia_no_clasificada,sequia_prev90_no_clasificada,clima_ok,prev90_ok,altitud_ok,hidro_ok,siniestro_ok
0,2016,1,1,PV,Riego,Aguascalientes,Aguascalientes,2247.15,300.0,0.0,300.0,7805071.04,3473.32,3473.32,7.490500,0.000000,0,0,21.546776,14.091694,29.001858,0.0,2113.060,21.827541,16.758167,15.601389,16.496674,-0.895285,21.587001,-0.040225,-0.137596,2120.581250,-7.521250,0.609252,-0.767370,183.0,183.0,90.0,False,False,1921.8,1795.0,2093.0,82.7,0,1,-29.64,249.77,0,False,True,True,True,True,True,True
1,2016,1,1,PV,Temporal,Aguascalientes,Aguascalientes,3350.83,5032.0,13.0,5045.0,11255639.02,3359.06,3359.06,0.665904,0.002577,0,0,21.546776,14.091694,29.001858,0.0,2113.060,21.827541,16.758167,15.601389,16.496674,-0.895285,21.587001,-0.040225,-0.137596,2120.581250,-7.521250,0.609252,-0.767370,183.0,183.0,90.0,False,False,1921.8,1795.0,2093.0,82.7,0,1,-29.64,249.77,0,False,True,True,True,True,True,True
2,2016,1,2,PV,Riego,Aguascalientes,Asientos,3340.00,630.0,0.0,630.0,11690000.00,3500.00,3500.00,5.301587,0.000000,0,0,20.726667,13.286612,28.166721,0.0,1962.980,21.115000,15.843750,14.637056,15.675924,-1.038868,20.852729,-0.126062,-0.390252,1986.326875,-23.346875,0.516055,-0.931624,183.0,183.0,90.0,False,False,2038.4,1973.0,2129.0,43.4,0,1,-29.64,249.77,0,False,True,True,True,True,True,True
3,2016,1,2,PV,Temporal,Aguascalientes,Asientos,2770.00,4525.0,0.0,4525.0,9695000.00,3500.00,3500.00,0.612155,0.000000,0,0,20.726667,13.286612,28.166721,0.0,1962.980,21.115000,15.843750,14.637056,15.675924,-1.038868,20.852729,-0.126062,-0.390252,1986.326875,-23.346875,0.516055,-0.931624,183.0,183.0,90.0,False,False,2038.4,1973.0,2129.0,43.4,0,1,-29.64,249.77,0,False,True,True,True,True,True,True
4,2016,1,3,PV,Riego,Aguascalientes,Calvillo,333.00,58.0,0.0,58.0,1251800.28,3759.16,3759.16,5.741379,0.000000,0,0,22.169973,14.747322,29.592623,0.0,2227.105,22.262951,17.324167,16.173333,16.990083,-0.816750,22.196363,-0.026390,-0.093613,2232.002500,-4.897500,0.602008,-0.656086,183.0,183.0,90.0,False,False,1975.9,1563.0,2402.0,262.0,0,1,-29.64,249.77,0,False,True,True,True,True,True,True


In [225]:
# --- Diagnóstico de cobertura del join y guardado ----------------------------
n = len(df_modelo)
print(f'filas totales (municipio-año-ciclo, 2016-2024): {n}')
print(f'  años      : {sorted(df_modelo.anio.unique())}')
print(f'  ciclos    : {df_modelo.nomcicloproductivo.value_counts().to_dict()}')
print(f'  modalidad : {df_modelo.nommodalidad.value_counts().to_dict()}')

print('\nnivel_sequia_max / nivel_sequia_prev90_max (tras imputar NaN->0):')
print(pd.concat({'ciclo':  df_modelo['nivel_sequia_max'].value_counts().sort_index(),
                 'prev90': df_modelo['nivel_sequia_prev90_max'].value_counts().sort_index()},
                axis=1).fillna(0).astype(int).to_string())

print('\nconfusores Daymet faltantes por año x ciclo (% filas sin tmean_ciclo):')
print(df_modelo.assign(na=df_modelo['tmean_ciclo'].isna())
      .pivot_table(index='anio', columns='nomcicloproductivo', values='na', aggfunc='mean').round(3))

print('\nprev90 incompleto por año x ciclo (% filas):')
print(df_modelo.pivot_table(index='anio', columns='nomcicloproductivo',
                            values='prev90_incompleta', aggfunc='mean').round(3))

print('\naltitud municipal:')
_sin_alt = df_modelo['altitud_media_m'].isna()
print(f'  filas con altitud   : {(~_sin_alt).sum()} ({(~_sin_alt).mean():.1%})')
print(f'  municipios sin altitud: {df_modelo.loc[_sin_alt, ["idestado", "idmunicipio"]].drop_duplicates().shape[0]}')
print(df_modelo[['altitud_media_m', 'altitud_std_m']].describe().round(1).to_string())

print('\nestrés hídrico (cuenca/acuífero):')
_sin_ca = df_modelo['acuifero_disp'].isna()
print(f'  filas con dato        : {(~_sin_ca).sum()} ({(~_sin_ca).mean():.1%})')
print(f'  municipios sin dato   : {df_modelo.loc[_sin_ca, ["idestado", "idmunicipio"]].drop_duplicates().shape[0]}')
print('  tiene_acuifero_sobreexplotado:',
      df_modelo['tiene_acuifero_sobreexplotado'].value_counts(dropna=False).sort_index().to_dict())
print('  tiene_cuenca_sin_disp        :',
      df_modelo['tiene_cuenca_sin_disp'].value_counts(dropna=False).sort_index().to_dict())
print('  acuifero_condicion_ok=0 ("Condición" no publicada esa fuente):',
      int((df_modelo['acuifero_condicion_ok'] == 0).sum()),
      f"({(df_modelo['acuifero_condicion_ok'] == 0).mean():.1%})  -> tiene_acuifero_sobreexplotado no confiable ahí")
print(df_modelo[['acuifero_disp', 'cuencas_disp']].describe().round(1).to_string())

print('\ntasa de siniestro (siniestrada/sembrada) y punto de corte de Y:')
print(f'  umbral (fijo): {UMBRAL_SINIESTRO:.3f}')
print(f'  siniestro_ok=False (tasa_siniestro >= {UMBRAL_SINIESTRO:.0%}, fuera de Y): '
      f'{int((~df_modelo["siniestro_ok"]).sum())} ({(~df_modelo["siniestro_ok"]).mean():.1%})')

print('\nfilas completas para el modelo (clima_ok & prev90_ok & altitud_ok & hidro_ok & siniestro_ok & sequía clasificada):')
apto = (df_modelo['clima_ok'] & df_modelo['prev90_ok'] & df_modelo['altitud_ok']
        & df_modelo['hidro_ok'] & df_modelo['siniestro_ok'] & ~df_modelo['sequia_no_clasificada'])
print(f'  {apto.sum()} de {n} ({apto.mean():.1%})  |  municipios: '
      f'{df_modelo.loc[apto, ["idestado", "idmunicipio"]].drop_duplicates().shape[0]}')

RUTA_MODELO = RUTA_PROC / 'tabla_modelo.csv'
df_modelo.to_csv(RUTA_MODELO, index=False, encoding='utf-8')
print(f'\nguardado -> {RUTA_MODELO}  {df_modelo.shape}')
df_modelo[LLAVES + ['nommodalidad', 'nomestado', 'nommunicipio', 'sembrada', 'rendimiento_real',
                    'nivel_sequia_max', 'nivel_sequia_prev90_max', 'sequia_no_clasificada',
                    'tmean_ciclo_anom', 'dias_helada', 'gdd_anom',
                    'tmean_inicio_anom', 'tmean_preseason_anom', 'tmean_prev90_anom',
                    'tmean_normal', 'tmean_prev90_normal',
                    'altitud_media_m', 'altitud_std_m',
                    'tiene_acuifero_sobreexplotado', 'tiene_cuenca_sin_disp', 'acuifero_disp', 'cuencas_disp',
                    'acuifero_condicion_ok', 'tasa_siniestro', 'siniestro_ok',
                    'clima_ok', 'prev90_ok', 'altitud_ok', 'hidro_ok']].head(15)

filas totales (municipio-año-ciclo, 2016-2024): 40267
  años      : [np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024)]
  ciclos    : {'PV': 29488, 'OI': 10779}
  modalidad : {'Temporal': 24860, 'Riego': 15407}

nivel_sequia_max / nivel_sequia_prev90_max (tras imputar NaN->0):
   ciclo  prev90
0  16847   24991
1  12519    8653
2   6677    4598
3   3559    1711
4    665     314

confusores Daymet faltantes por año x ciclo (% filas sin tmean_ciclo):
nomcicloproductivo     OI     PV
anio                            
2016                0.005  0.002
2017                0.005  0.002
2018                0.005  0.002
2019                0.004  0.002
2020                0.004  0.002
2021                0.004  0.002
2022                0.005  0.002
2023                0.005  0.002
2024                0.004  0.002

prev90 incompleto por año x ciclo (% filas):
nomcicloproductivo   OI   PV
anio             

,idestado,idmunicipio,anio,nomcicloproductivo,nommodalidad,nomestado,nommunicipio,sembrada,rendimiento_real,nivel_sequia_max,nivel_sequia_prev90_max,sequia_no_clasificada,tmean_ciclo_anom,dias_helada,gdd_anom,tmean_inicio_anom,tmean_preseason_anom,tmean_prev90_anom,tmean_normal,tmean_prev90_normal,altitud_media_m,altitud_std_m,tiene_acuifero_sobreexplotado,tiene_cuenca_sin_disp,acuifero_disp,cuencas_disp,acuifero_condicion_ok,tasa_siniestro,siniestro_ok,clima_ok,prev90_ok,altitud_ok,hidro_ok
0,1,1,2016,PV,Riego,Aguascalientes,Aguascalientes,300.0,7.490500,0,0,False,-0.040225,0.0,-7.521250,0.609252,-0.767370,-0.895285,21.587001,16.496674,1921.8,82.7,0,1,-29.64,249.77,0,0.000000,True,True,True,True,True
1,1,1,2016,PV,Temporal,Aguascalientes,Aguascalientes,5045.0,0.665904,0,0,False,-0.040225,0.0,-7.521250,0.609252,-0.767370,-0.895285,21.587001,16.496674,1921.8,82.7,0,1,-29.64,249.77,0,0.002577,True,True,True,True,True
2,1,2,2016,PV,Riego,Aguascalientes,Asientos,630.0,5.301587,0,0,False,-0.126062,0.0,-23.346875,0.516055,-0.931624,-1.038868,20.852729,15.675924,2038.4,43.4,0,1,-29.64,249.77,0,0.000000,True,True,True,True,True
3,1,2,2016,PV,Temporal,Aguascalientes,Asientos,4525.0,0.612155,0,0,False,-0.126062,0.0,-23.346875,0.516055,-0.931624,-1.038868,20.852729,15.675924,2038.4,43.4,0,1,-29.64,249.77,0,0.000000,True,True,True,True,True
4,1,3,2016,PV,Riego,Aguascalientes,Calvillo,58.0,5.741379,0,0,False,-0.026390,0.0,-4.897500,0.602008,-0.656086,-0.816750,22.196363,16.990083,1975.9,262.0,0,1,-29.64,249.77,0,0.000000,True,True,True,True,True
5,1,3,2016,PV,Temporal,Aguascalientes,Calvillo,1370.0,0.672774,0,0,False,-0.026390,0.0,-4.897500,0.602008,-0.656086,-0.816750,22.196363,16.990083,1975.9,262.0,0,1,-29.64,249.77,0,0.000000,True,True,True,True,True
6,1,4,2016,PV,Riego,Aguascalientes,Cosío,640.0,7.526562,0,0,False,-0.189740,0.0,-34.984375,0.413402,-0.981851,-1.101812,20.977199,15.633868,2029.7,91.9,0,1,-29.64,249.77,0,0.000000,True,True,True,True,True
7,1,4,2016,PV,Temporal,Aguascalientes,Cosío,1250.0,0.587200,0,0,False,-0.189740,0.0,-34.984375,0.413402,-0.981851,-1.101812,20.977199,15.633868,2029.7,91.9,0,1,-29.64,249.77,0,0.000000,True,True,True,True,True
8,1,5,2016,PV,Riego,Aguascalientes,Jesús María,266.0,7.258835,0,0,False,-0.083193,0.0,-15.450000,0.564314,-0.851395,-0.966840,21.094286,16.046618,2042.6,125.6,0,1,-29.64,249.77,0,0.000000,True,True,True,True,True
9,1,5,2016,PV,Temporal,Aguascalientes,Jesús María,630.0,0.580016,0,0,False,-0.083193,0.0,-15.450000,0.564314,-0.851395,-0.966840,21.094286,16.046618,2042.6,125.6,0,1,-29.64,249.77,0,0.000000,True,True,True,True,True


In [226]:
df_modelo

,anio,idestado,idmunicipio,nomcicloproductivo,nommodalidad,nomestado,nommunicipio,volumenproduccion,cosechada,siniestrada,sembrada,valorproduccion,precio_mean,precio_total,rendimiento_real,tasa_siniestro,nivel_sequia_max,nivel_sequia_prev90_max,tmean_ciclo,tmin_ciclo_mean,tmax_ciclo_mean,dias_helada,gdd_ciclo,tmean_inicio,tmean_preseason,tmean_prev90,tmean_prev90_normal,tmean_prev90_anom,tmean_normal,tmean_ciclo_anom,tmean_ciclo_anom_z,gdd_normal,gdd_anom,tmean_inicio_anom,tmean_preseason_anom,n_dias_ventana,n_dias_con_dato,n_dias_prev90,ventana_incompleta,prev90_incompleta,altitud_media_m,altitud_min_m,altitud_max_m,altitud_std_m,tiene_acuifero_sobreexplotado,tiene_cuenca_sin_disp,acuifero_disp,cuencas_disp,acuifero_condicion_ok,sequia_no_clasificada,sequia_prev90_no_clasificada,clima_ok,prev90_ok,altitud_ok,hidro_ok,siniestro_ok
0,2016,1,1,PV,Riego,Aguascalientes,Aguascalientes,2247.15,300.0,0.0,300.0,7805071.04,3473.32,3473.32,7.490500,0.000000,0,0,21.546776,14.091694,29.001858,0.0,2113.060,21.827541,16.758167,15.601389,16.496674,-0.895285,21.587001,-0.040225,-0.137596,2120.581250,-7.521250,0.609252,-0.767370,183.0,183.0,90.0,False,False,1921.8,1795.0,2093.0,82.7,0,1,-29.64,249.77,0,False,True,True,True,True,True,True
1,2016,1,1,PV,Temporal,Aguascalientes,Aguascalientes,3350.83,5032.0,13.0,5045.0,11255639.02,3359.06,3359.06,0.665904,0.002577,0,0,21.546776,14.091694,29.001858,0.0,2113.060,21.827541,16.758167,15.601389,16.496674,-0.895285,21.587001,-0.040225,-0.137596,2120.581250,-7.521250,0.609252,-0.767370,183.0,183.0,90.0,False,False,1921.8,1795.0,2093.0,82.7,0,1,-29.64,249.77,0,False,True,True,True,True,True,True
2,2016,1,2,PV,Riego,Aguascalientes,Asientos,3340.00,630.0,0.0,630.0,11690000.00,3500.00,3500.00,5.301587,0.000000,0,0,20.726667,13.286612,28.166721,0.0,1962.980,21.115000,15.843750,14.637056,15.675924,-1.038868,20.852729,-0.126062,-0.390252,1986.326875,-23.346875,0.516055,-0.931624,183.0,183.0,90.0,False,False,2038.4,1973.0,2129.0,43.4,0,1,-29.64,249.77,0,False,True,True,True,True,True,True
3,2016,1,2,PV,Temporal,Aguascalientes,Asientos,2770.00,4525.0,0.0,4525.0,9695000.00,3500.00,3500.00,0.612155,0.000000,0,0,20.726667,13.286612,28.166721,0.0,1962.980,21.115000,15.843750,14.637056,15.675924,-1.038868,20.852729,-0.126062,-0.390252,1986.326875,-23.346875,0.516055,-0.931624,183.0,183.0,90.0,False,False,2038.4,1973.0,2129.0,43.4,0,1,-29.64,249.77,0,False,True,True,True,True,True,True
4,2016,1,3,PV,Riego,Aguascalientes,Calvillo,333.00,58.0,0.0,58.0,1251800.28,3759.16,3759.16,5.741379,0.000000,0,0,22.169973,14.747322,29.592623,0.0,2227.105,22.262951,17.324167,16.173333,16.990083,-0.816750,22.196363,-0.026390,-0.093613,2232.002500,-4.897500,0.602008,-0.656086,183.0,183.0,90.0,False,False,1975.9,1563.0,2402.0,262.0,0,1,-29.64,249.77,0,False,True,True,True,True,True,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
40262,2024,32,56,PV,Riego,Zacatecas,Zacatecas,6803.40,870.0,0.0,870.0,53645200.00,7550.00,15100.00,7.820000,0.000000,2,3,21.070956,13.558852,28.583060,0.0,2025.985,22.071639,15.607333,14.815222,14.711681,0.103542,20.054863,1.016093,2.455207,1840.668125,185.316875,2.178494,-0.305705,183.0,183.0,90.0,False,False,2302.7,2193.0,2481.0,98.1,1,1,-10.38,95.20,1,False,False,True,True,True,True,True
40263,2024,32,56,PV,Temporal,Zacatecas,Zacatecas,574.00,700.0,0.0,700.0,3478440.00,6060.00,6060.00,0.820000,0.000000,2,3,21.070956,13.558852,28.583060,0.0,2025.985,22.071639,15.607333,14.815222,14.711681,0.103542,20.054863,1.016093,2.455207,1840.668125,185.316875,2.178494,-0.305705,183.0,183.0,90.0,False,False,2302.7,2193.0,2481.0,98.1,1,1,-10.38,95.20,1,False,False,True,True,True,True,True
40264,2024,32,57,PV,Riego,Zacatecas,Trancoso,4232.80,520.0,0.0,520.0,32486740.00,7450.00,14900.00,8.140000,0.000000,1,3,21.685601,14.316831,29.054372,0.0,21

In [227]:
df_modelo.columns

Index(['anio', 'idestado', 'idmunicipio', 'nomcicloproductivo', 'nommodalidad',
       'nomestado', 'nommunicipio', 'volumenproduccion', 'cosechada',
       'siniestrada', 'sembrada', 'valorproduccion', 'precio_mean',
       'precio_total', 'rendimiento_real', 'tasa_siniestro',
       'nivel_sequia_max', 'nivel_sequia_prev90_max', 'tmean_ciclo',
       'tmin_ciclo_mean', 'tmax_ciclo_mean', 'dias_helada', 'gdd_ciclo',
       'tmean_inicio', 'tmean_preseason', 'tmean_prev90',
       'tmean_prev90_normal', 'tmean_prev90_anom', 'tmean_normal',
       'tmean_ciclo_anom', 'tmean_ciclo_anom_z', 'gdd_normal', 'gdd_anom',
       'tmean_inicio_anom', 'tmean_preseason_anom', 'n_dias_ventana',
       'n_dias_con_dato', 'n_dias_prev90', 'ventana_incompleta',
       'prev90_incompleta', 'altitud_media_m', 'altitud_min_m',
       'altitud_max_m', 'altitud_std_m', 'tiene_acuifero_sobreexplotado',
       'tiene_cuenca_sin_disp', 'acuifero_disp', 'cuencas_disp',
       'acuifero_condicion_ok', 'sequia_n

In [228]:
df_modelo[['rendimiento_real']].max()

rendimiento_real    13.75
dtype: float64